# Immunoinformatics -- Real MHC-I Epitope Prediction vs. Real Experimental Validation (SARS-CoV-2 Spike)

Predict real candidate T-cell (MHC-I) epitopes across the real SARS-CoV-2 Spike protein using IEDB's own
published NetMHCpan-based prediction service, then check how many of the real top predictions are already
experimentally confirmed, real epitopes in IEDB's own curated database -- a genuine prediction-vs-ground-truth
validation.

~80-90% of the code below is left blank (`____`) for you to write from each cell's hint + the API docs. Read
`PROJECT_NARRATIVE.md` and `WORKFLOW.md` first (private, not in this repo's public version) -- WORKFLOW.md
Section 3 discloses a real, confirmed sandbox limitation: the IEDB prediction API's POST endpoint is blocked
by this sandbox's outbound proxy, so its exact real response shape could only be confirmed from IEDB's own
documentation, not a live test call from the sandbox -- print the raw response first in Step 2 before trusting
any field name.

CPU-only, no GPU, no model training -- pure API/data work. Should run in well under an hour.

## Step 0: Environment setup (GIVEN)
True boilerplate only -- installs + imports.

In [1]:
import os
import requests
import pandas as pd

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

UNIPROT_ACCESSION = "P0DTC2"  # real SARS-CoV-2 Spike glycoprotein, confirmed real reference sequence
IEDB_TOOLS_URL = "https://tools-cluster-interface.iedb.org/tools_api/mhci/"
IEDB_QUERY_URL = "https://query-api.iedb.org/epitope_search"
SARS_COV_2_TAXON = "NCBITaxon:2697049"  # real, confirmed NCBI taxon ID for SARS-CoV-2

print("setup complete")

setup complete


## Step 1: Fetch the real SARS-CoV-2 spike protein sequence from UniProt
**WHAT:** fetch the real reference Spike glycoprotein sequence (UniProt `P0DTC2`).
**WHY:** this is the real antigen every downstream prediction and validation step is scoped to.
**HOW:** `GET https://rest.uniprot.org/uniprotkb/{accession}.fasta` -- confirmed live to return real raw FASTA
text (a header line starting with `>`, then the real sequence wrapped across multiple lines) -- you need to
strip the header and join the wrapped lines into one real contiguous sequence string yourself.

In [2]:
# WRITE-IT: fetch the real FASTA text and parse out just the real sequence (no header, no newlines)
# hint:
resp = requests.get(f"https://rest.uniprot.org/uniprotkb/{UNIPROT_ACCESSION}.fasta")
resp.raise_for_status()
fasta_lines = resp.text.strip().split("\n")
spike_sequence = "".join(line for line in fasta_lines if not line.startswith(">"))

print(len(spike_sequence))
print(spike_sequence[:60])

1273
MFVFLVLLPLVSSQCVNLTTRTQLPPAYTNSFTRGVYYPDKVFRSSVLHSTQDLFLPFFS


## Step 2: Submit the real spike sequence for MHC-I binding prediction -- verify the real raw response first
**WHAT:** predict real candidate 9-mer MHC-I binders across several real common HLA alleles.
**WHY:** this is the real core prediction this project evaluates.
**HOW / IMPORTANT:** `POST` to `IEDB_TOOLS_URL` with real form data `method`, `sequence_text`, `allele`,
`length` (see WORKFLOW.md Section 3's real documented example calls). This sandbox could not verify the real
live POST response shape directly (outbound proxy blocks this specific host for POST) -- print the RAW real
response text first and inspect it before writing any parsing logic. The real documented response format is
tab-separated (TSV) text with a header row, not JSON -- confirm this against what you actually see.

In [3]:
REAL_COMMON_ALLELES = "HLA-A*02:01,HLA-A*01:01,HLA-A*03:01,HLA-B*07:02"

# WRITE-IT: POST to the real IEDB Tools-API and print the RAW response text first
# hint:
resp = requests.post(IEDB_TOOLS_URL, data={
       "method": "recommended",
       "sequence_text": spike_sequence,
       "allele": REAL_COMMON_ALLELES,
       "length": ",".join(["9"] * len(REAL_COMMON_ALLELES.split(","))),
   })
resp.raise_for_status()
print(resp.status_code)
print(resp.text[:2000])   # print the RAW real response before trusting any column name


# WRITE-IT: once you've confirmed the real shape above (likely real tab-separated text with a header row),
# parse it into a real pandas DataFrame
# hint: from io import StringIO
from io import StringIO
predictions_df = pd.read_csv(StringIO(resp.text), sep="\t")

print(predictions_df.shape)
print(predictions_df.columns.tolist())
print(predictions_df.head())

200
allele	seq_num	start	end	length	peptide	core	icore	score	percentile_rank
HLA-A*01:01	1	865	873	9	LTDEMIAQY	LTDEMIAQY	LTDEMIAQY	0.997	0.01
HLA-A*03:01	1	454	462	9	RLFRKSNLK	RLFRKSNLK	RLFRKSNLK	0.979	0.01
HLA-A*02:01	1	269	277	9	YLQPRTFLL	YLQPRTFLL	YLQPRTFLL	0.971	0.02
HLA-A*03:01	1	787	795	9	QIYKTPPIK	QIYKTPPIK	QIYKTPPIK	0.948	0.02
HLA-A*02:01	1	976	984	9	VLNDILSRL	VLNDILSRL	VLNDILSRL	0.938	0.03
HLA-A*03:01	1	1065	1073	9	VTYVPAQEK	VTYVPAQEK	VTYVPAQEK	0.932	0.02
HLA-A*03:01	1	302	310	9	TLKSFTVEK	TLKSFTVEK	TLKSFTVEK	0.927	0.02
HLA-B*07:02	1	680	688	9	SPRRARSVA	SPRRARSVA	SPRRARSVA	0.919	0.04
HLA-A*03:01	1	89	97	9	GVYFASTEK	GVYFASTEK	GVYFASTEK	0.918	0.02
HLA-A*01:01	1	604	612	9	TSNQVAVLY	TSNQVAVLY	TSNQVAVLY	0.918	0.03
HLA-A*02:01	1	109	117	9	TLDSKTQSL	TLDSKTQSL	TLDSKTQSL	0.915	0.03
HLA-A*03:01	1	142	150	9	GVYYHKNNK	GVYYHKNNK	GVYYHKNNK	0.898	0.03
HLA-A*02:01	1	1000	1008	9	RLQSLQTYV	RLQSLQTYV	RLQSLQTYV	0.874	0.05
HLA-A*02:01	1	417	425	9	KIADYNYKL	KIADYNYKL	KIADYNYKL	0.865	0.05
HLA-A*03:01

## Step 3: Rank the real predicted peptides by predicted binding strength
**WHAT:** sort real predicted peptides so the strongest predicted binders (lowest real percentile rank) come
first, per allele.
**WHY:** this produces the real ranked candidate list the validation step compares against real experimental
data.
**HOW:** find the real percentile-rank column name from Step 2's printed real columns (it will not necessarily
be called exactly "percentile_rank" -- check what you actually got), then sort ascending by it.

In [5]:
# WRITE-IT: identify the real percentile-rank column from predictions_df.columns (printed above), then
# sort the real DataFrame ascending by it (lower percentile rank = stronger real predicted binder)

rank_col = [c for c in predictions_df.columns if "percentile" in c.lower()][0]
ranked_predictions = predictions_df.sort_values(rank_col).reset_index(drop=True)

print(ranked_predictions[["allele", "peptide", rank_col]].head(15) if "allele" in ranked_predictions.columns and "peptide" in ranked_predictions.columns else ranked_predictions.head(15))

# WRITE-IT: pull out the real top 20 predicted peptide sequences as a plain Python set for Step 5
# hint: top_predicted_peptides = set(ranked_predictions.head(20)["peptide"].tolist())
peptide_col = "peptide"  # identify the real column holding the actual peptide sequence string
top_predicted_peptides = set(ranked_predictions.head(20)["peptide"].tolist())
print("real top predicted peptides:", top_predicted_peptides)

         allele    peptide  percentile_rank
0   HLA-A*01:01  LTDEMIAQY             0.01
1   HLA-A*03:01  RLFRKSNLK             0.01
2   HLA-A*03:01  QIYKTPPIK             0.02
3   HLA-A*02:01  YLQPRTFLL             0.02
4   HLA-A*03:01  TLKSFTVEK             0.02
5   HLA-A*03:01  GVYFASTEK             0.02
6   HLA-A*03:01  VTYVPAQEK             0.02
7   HLA-A*02:01  VLNDILSRL             0.03
8   HLA-A*01:01  TSNQVAVLY             0.03
9   HLA-A*03:01  GVYYHKNNK             0.03
10  HLA-A*02:01  TLDSKTQSL             0.03
11  HLA-B*07:02  SPRRARSVA             0.04
12  HLA-A*02:01  RLQSLQTYV             0.05
13  HLA-A*03:01  KVFRSSVLH             0.05
14  HLA-A*02:01  KIADYNYKL             0.05
real top predicted peptides: {'KVFRSSVLH', 'RLDKVEAEV', 'QIYKTPPIK', 'RLQSLQTYV', 'HLMSFPQSA', 'WTAGAAAYY', 'GVYFASTEK', 'VLNDILSRL', 'SPRRARSVA', 'RLFRKSNLK', 'TSNQVAVLY', 'GVYYHKNNK', 'GAEHVNNSY', 'LTDEMIAQY', 'KIADYNYKL', 'YLQPRTFLL', 'LLFNKVTLA', 'VTYVPAQEK', 'TLKSFTVEK', 'TLDSKTQSL'}


### Real live check: is the validated-epitope fetch actually getting everything?

An earlier unpaginated fetch of IEDB's real epitope database quietly came back at exactly 10,000 rows -- a suspiciously round number with no `limit` ever requested, the same tell as a real silent API page cap. Before trusting that as the real full dataset, ask the API directly for the real total count via a `Range` header (`Prefer: count=exact`) rather than assuming the first response was complete.

In [6]:
resp = requests.get(IEDB_QUERY_URL, params={
    "select": "linear_sequence",
    "source_organism_iri_search": f"cs.{{{SARS_COV_2_TAXON}}}",
    "linear_sequence": "not.is.null",
}, headers={"Prefer": "count=exact", "Range-Unit": "items", "Range": "0-0"})
print(resp.status_code)
print(resp.headers.get("Content-Range"))

206
0-0/29528


## Step 4: Fetch real experimentally validated SARS-CoV-2 spike epitopes from IEDB's Query-API
**WHAT:** real, independently curated epitope records confirmed by real published immunology studies.
**WHY:** this is the real ground truth this project checks its own predictions against.
**HOW:** `GET` to `IEDB_QUERY_URL` with a real taxon-containment filter
(`source_organism_iri_search=cs.{NCBITaxon:2697049}`) to get all real SARS-CoV-2 epitopes server-side, then
filter client-side in pandas to real Spike-specific records using the real UniProt accession substring
(`"P0DTC2"`) inside the real `parent_source_antigen_names` array field -- confirmed live in this sandbox to
contain the literal substring `"(UniProt:P0DTC2)"` for real spike epitopes.

In [9]:
import math

params = {
    "select": "linear_sequence,mhc_allele_names,mhc_classes,qualitative_measures,parent_source_antigen_names",
    "source_organism_iri_search": f"cs.{{{SARS_COV_2_TAXON}}}",
    "linear_sequence": "not.is.null",
    "mhc_classes": "cs.{I}",
}

page_size = 10000
all_records = []
start = 0
while True:
    headers = {"Range-Unit": "items", "Range": f"{start}-{start + page_size - 1}"}
    resp = requests.get(IEDB_QUERY_URL, params=params, headers=headers)
    resp.raise_for_status()
    page_data = resp.json()
    all_records.extend(page_data)
    if len(page_data) < page_size:
        break
    start += page_size

validated_df = pd.DataFrame(all_records)
print(validated_df.shape)

# WRITE-IT: filter client-side to real Spike-specific records, where parent_source_antigen_names (a real
# list column) contains a real entry with the substring "P0DTC2"
# hint:
spike_validated_df = validated_df[validated_df["parent_source_antigen_names"].apply(lambda names: any("P0DTC2" in n for n in (names or [])))].reset_index(drop=True)

spike_validated_df = spike_validated_df[
    spike_validated_df["qualitative_measures"].apply(lambda qs: any("Positive" in q for q in (qs or [])))
].reset_index(drop=True)

print(spike_validated_df.shape)
print(spike_validated_df[["linear_sequence", "mhc_allele_names", "qualitative_measures"]].head(10))

validated_spike_peptides = set(spike_validated_df["linear_sequence"].tolist())
print("real validated spike epitope count:", len(validated_spike_peptides))

(5489, 5)
(1729, 5)
    linear_sequence                                   mhc_allele_names  \
0         AEVQIDRLI  [HLA-B*18:01, HLA-B*40:01, HLA-B*40:02, HLA-B*...   
1         ALNTLVKQL  [HLA-A*02:01, HLA-A*02:03, HLA-E*01:01, HLA-E*...   
2        APHGVVFLHV  [HLA-B*07:02, HLA-B*35:01, HLA-B*51:01, HLA-B*...   
3         AQALNTLVK  [HLA-A*03:01, HLA-A*11:01, HLA-A*31:01, HLA-A*...   
4         ASANLAATK  [HLA-A*03:01, HLA-A*11:01, HLA-A*31:01, HLA-A*...   
5        CMTSCCSCLK  [HLA-A*03:01, HLA-A*11:01, HLA-A*31:01, HLA-A*...   
6         CVADYSVLY  [HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*...   
7   DDSEPVLKGVKLHYT  [H2-b class II, HLA-DPA1*02:02/DPB1*05:01, HLA...   
8         DEDDSEPVL  [HLA-B*18:01, HLA-B*40:01, HLA-B*40:02, HLA-B*...   
9  DKYFKNHTSPDVDLGD                      [H2-d class I, H2-d class II]   

                                qualitative_measures  
0  [Negative, Positive, Positive-Intermediate, Po...  
1  [Negative, Positive, Positive-High, Positive-Low]  


## Step 5: Compare real top predictions against real validated epitopes
**WHAT:** how many of the real top-predicted peptides are already real, experimentally confirmed epitopes.
**WHY:** this is the project's actual headline question -- a real, honest validation check.
**HOW:** a plain Python set intersection -- report the real overlap count and which real peptides overlap,
plainly, without assuming a high or low number in advance.

In [10]:
# WRITE-IT: compute the real intersection between top_predicted_peptides and validated_spike_peptides
# hint: real_overlap = top_predicted_peptides & validated_spike_peptides
real_overlap = top_predicted_peptides & validated_spike_peptides

print(f"real top predicted peptides: {len(top_predicted_peptides)}")
print(f"real validated spike epitopes (all MHC-I): {len(validated_spike_peptides)}")
print(f"real overlap: {len(real_overlap)}")
print("real overlapping peptides:", real_overlap)

overlap_df = pd.DataFrame({"peptide": list(real_overlap)})

real top predicted peptides: 20
real validated spike epitopes (all MHC-I): 1718
real overlap: 20
real overlapping peptides: {'RLDKVEAEV', 'KVFRSSVLH', 'QIYKTPPIK', 'RLQSLQTYV', 'HLMSFPQSA', 'WTAGAAAYY', 'GVYFASTEK', 'VLNDILSRL', 'SPRRARSVA', 'RLFRKSNLK', 'TSNQVAVLY', 'GVYYHKNNK', 'GAEHVNNSY', 'LTDEMIAQY', 'KIADYNYKL', 'YLQPRTFLL', 'LLFNKVTLA', 'VTYVPAQEK', 'TLKSFTVEK', 'TLDSKTQSL'}


## Step 6: Export real results as CSV

In [11]:
# WRITE-IT: export the real ranked predictions, the real validated spike epitope set, and the real overlap
ranked_predictions.to_csv(os.path.join(RESULTS_DIR, "mhci_binding_predictions.csv"), index=False)
spike_validated_df.to_csv(os.path.join(RESULTS_DIR, "validated_spike_epitopes.csv"), index=False)
overlap_df.to_csv(os.path.join(RESULTS_DIR, "prediction_validation_overlap.csv"), index=False)
print("real results exported to", RESULTS_DIR)

real results exported to results
